# Day 8: directly test a JEPA latent edit

**Question:** does moving an object's frozen JEPA representation express the requested displacement while retaining its appearance and the unselected object?

This notebook runs the paired-video experiment in `experiments/day8` from a fixed source commit. It contains **no saved execution outputs or claimed results**. It is a reproduction recipe, not evidence that this notebook has been run end to end.

The experiment uses 24 train, 8 development, and 16 independently seeded test scenes. Each has two textured moving balls; only one ball is translated. Eight test scenes use familiar shift magnitudes and eight use held-out magnitudes. The frozen official V-JEPA 2.1 ViT-L/384 produces 1024-dimensional tokens. Training fits two small correction arms with three paired seeds, plus one separately fitted token-only occupancy/RGB probe. No SAM or other segmentation model is used.

**Compute:** choose a Colab GPU runtime when available. GPU and TPU allocation were quota-blocked in the active session on 9 October 2026, so this recipe also supports CPU. CPU extraction is slow; the selected model's official checkpoint is several gigabytes. CPU defaults to float32 inference. CPU bfloat16 was validated against a float32 reference with the same weights on the local 8-core CPU; the Colab CPU was not validated. This recipe therefore keeps Colab CPU extraction at float32. TPU execution is not enabled here. CUDA defaults to float16 inference. Every run records its actual backend and precision; do not mix them in one cache.

**What this can establish:** controlled coarse latent editability in separated-lane synthetic scenes with oracle masks throughout the video. The probe is a 24×24 patch readout, **not a full-resolution video decoder**. The current backbone attends offline within two independent 16-frame blocks. This does not yet demonstrate faces, car parts, occlusion, 3-D reasoning, or photorealistic generation.

## 1. Configuration and exact source checkout

The source revision must be a full commit SHA containing the complete Day 8 pipeline. The repository branch is fetched only to locate its history; the actual checkout is detached at the pinned commit. Reusing a completed cache or checkpoint requires exact matching hashes. If changing code, compute backend, precision, or protocol, use a new run directory.

`FREEZE_COMMIT` and `FREEZE_REPO_PATH` identify the Git-published pretest freeze. They are verified only after training, so the full setup and training can run before that commit exists. A fresh reproduction must publish its own completed `training/checkpoint_freeze.json`; an earlier run's freeze can be used only if its bytes match exactly. All test extraction requires the publication check below.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, re, shutil, subprocess, sys, zipfile

CODE_COMMIT = "9ebd665f1be56535199af13d2f6deb8c27af92b1"
FREEZE_COMMIT = "598f5bc1770c72b4af437a44f834cef41f0aeb08"
FREEZE_REPO_PATH = "experiments/day8/run_2026-10-09/checkpoint_freeze.json"
REPO_URL = "https://github.com/skbdps/vjepa-poc.git"
REPO_BRANCH = "experiment/part-consistency-colab"
UPSTREAM_URL = "https://github.com/facebookresearch/vjepa2.git"
UPSTREAM_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"
REQUESTED_DEVICE = "auto"  # "auto", "cuda", or "cpu"
CPU_PRECISION = "float32"  # local CPU bfloat16 validation does not validate the Colab CPU
CUDA_PRECISION = "float16"  # use a new run directory if changing precision
RUN_NAME = "day8_direct_latent_edit"
THREADS = 2

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd() / "day8_workspace"
REPO = BASE_DIR / "vjepa-poc-day8"
UPSTREAM = BASE_DIR / "vjepa2-day8"
RUN = BASE_DIR / RUN_NAME
FEATURES, TRAINING, TEST, ANALYSIS = (RUN / name for name in ("features", "training", "test", "analysis"))
LOGS = RUN / "logs"
EXP = REPO / "experiments/day8"
FREEZE = TRAINING / "checkpoint_freeze.json"
for folder in (BASE_DIR, RUN, LOGS):
    folder.mkdir(parents=True, exist_ok=True)
CHILD_ENV = dict(os.environ, OMP_NUM_THREADS=str(THREADS), MKL_NUM_THREADS=str(THREADS))
assert re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT), "Replace CODE_COMMIT with the published Day 8 source SHA."
assert REQUESTED_DEVICE in ("auto", "cuda", "cpu")
assert CPU_PRECISION == "float32", "This recipe keeps the unvalidated Colab CPU on float32 inference."
assert CUDA_PRECISION in ("float16", "float32")

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def checkout_exact(url, folder, revision, branch=None):
    if (folder / ".git").exists():
        dirty = subprocess.check_output(["git", "-C", str(folder), "status", "--porcelain"], text=True).strip()
        if dirty:
            raise RuntimeError(f"Checkout contains local changes; preserve them before using this notebook: {folder}")
    else:
        command = ["git", "clone", "--filter=blob:none", "--no-checkout"]
        if branch:
            command += ["--branch", branch]
        subprocess.run(command + [url, str(folder)], check=True, env=CHILD_ENV)
    subprocess.run(["git", "-C", str(folder), "fetch", "origin", revision], check=True, env=CHILD_ENV)
    subprocess.run(["git", "-C", str(folder), "checkout", "--detach", revision], check=True, env=CHILD_ENV)
    actual = subprocess.check_output(["git", "-C", str(folder), "rev-parse", "HEAD"], text=True).strip()
    assert actual == revision

checkout_exact(REPO_URL, REPO, CODE_COMMIT, REPO_BRANCH)
checkout_exact(UPSTREAM_URL, UPSTREAM, UPSTREAM_COMMIT)
print("Experiment source:", CODE_COMMIT)
print("Official encoder source:", UPSTREAM_COMMIT)
print("Run directory:", RUN)

## 2. Dependencies and backend

Use Colab's matching PyTorch/torchvision installation. Install only the official backbone's inference dependencies needed by this experiment (`timm`, `einops`, NumPy), plus Pillow/Matplotlib and FFmpeg for scientific displays. The full upstream training/robotics dependency set is unnecessary here. The extractor verifies the official model checkpoint SHA-256 before loading it.

The actual package versions, source revisions, available memory/disk, and compute choice are recorded. CPU and CUDA runs can differ numerically; compare their measured results rather than assuming bitwise identity. Feature caches store float16 in both cases, while the recorded inference precision remains explicit.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import version
if find_spec("torch") is None or find_spec("torchvision") is None:
    raise RuntimeError("This recipe expects the matching PyTorch/torchvision stack supplied by Colab. Install a matching stack for a local runtime first.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm==1.0.26", "einops==0.8.2", "numpy", "pillow", "matplotlib"],
               check=True, env=CHILD_ENV)
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True, env=CHILD_ENV)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True, env=CHILD_ENV)
import torch
import numpy as np
DEVICE = ("cuda" if torch.cuda.is_available() else "cpu") if REQUESTED_DEVICE == "auto" else REQUESTED_DEVICE
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but is unavailable. Select CPU explicitly or obtain a GPU runtime.")
PRECISION = CUDA_PRECISION if DEVICE == "cuda" else CPU_PRECISION
torch.set_num_threads(THREADS)
source_files = ["data.py", "extract.py", "operators.py", "train.py", "probe.py", "evaluate.py", "visualize.py", "PROTOCOL.md"]
metadata = {
    "source_revision": CODE_COMMIT, "upstream_revision": UPSTREAM_COMMIT,
    "created_utc": datetime.now(timezone.utc).isoformat(), "python": platform.python_version(),
    "device": DEVICE, "inference_precision": PRECISION, "cache_precision": "float16",
    "gpu": torch.cuda.get_device_name() if DEVICE == "cuda" else None,
    "packages": {name: version(name) for name in ("torch", "torchvision", "numpy", "timm", "einops", "pillow", "matplotlib")},
    "child_thread_limits": THREADS,
    "source_hashes": {name: sha256(EXP / name) for name in source_files},
    "free_disk_gib_at_setup": shutil.disk_usage(BASE_DIR).free / 1024**3,
    "notebook_status": "source-only reproduction recipe; execution evidence comes from this run's logs and artifacts",
}
manifest_path = RUN / "notebook_manifest.json"
if manifest_path.exists():
    previous = json.loads(manifest_path.read_text())
    for key in ("source_revision", "upstream_revision", "device", "inference_precision", "source_hashes"):
        assert previous[key] == metadata[key], f"Existing run mismatch: {key}; use a new RUN_NAME."
    metadata["first_setup_utc"] = previous.get("first_setup_utc", previous["created_utc"])
manifest_path.write_text(json.dumps(metadata, indent=2) + "\n")
(RUN / "pip_freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print(json.dumps(metadata, indent=2))

def run_logged(command, name):
    command = list(map(str, command))
    print("Running:", " ".join(command), flush=True)
    with (LOGS / f"{name}.log").open("a") as log:
        log.write("\nRUN_STARTED_UTC " + datetime.now(timezone.utc).isoformat() + "\n")
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, env=CHILD_ENV)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
        log.write(f"RUN_EXIT_CODE {returncode}\n")
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

## 3. Implementation checks without opening test scenes

The renderer check uses training fixtures. The probe check uses independent synthetic features, not experiment videos. The evaluator check uses analytic arrays. Training additionally checks finite gradients, exact no-op behavior, and strict edit-support preservation before fitting the heads.

In [ ]:
run_logged([sys.executable, "-u", EXP / "data.py", "--self-check", "--output", RUN / "data_self_check.json"], "data_self_check")
run_logged([sys.executable, "-u", EXP / "probe.py", "--self-check"], "probe_self_check")
run_logged([sys.executable, "-u", EXP / "evaluate.py", "--self-check", "--out", RUN / "implementation_checks"], "evaluation_self_check")

## 4. Encode training and development videos only

Extract all 24 training and 8 development pairs. Each source and its genuine edited counterpart is encoded with the same frozen backbone. Operators later receive source features, both source object masks, and the requested displacement; they never receive target features as inference inputs.

Exact completed caches can be reused after a timeout. The manifest rejects altered source files and mixed backends/precision. A fresh training run refuses a directory that already contains test caches. A previously frozen run may be resumed without retraining or changing model selection.

In [ ]:
run_logged([sys.executable, "-u", EXP / "extract.py", "--out", FEATURES,
            "--upstream", UPSTREAM, "--splits", "train", "dev", "--device", DEVICE,
            "--precision", PRECISION], "train_dev_extraction")
feature_manifest = json.loads((FEATURES / "manifest.json").read_text())
counts = {split: sum(row["spec"]["split"] == split for row in feature_manifest["clips"]) for split in ("train", "dev", "test")}
assert counts["train"] == 24 and counts["dev"] == 8
if counts["test"]:
    assert FREEZE.exists(), "Test data exists without a completed pretest freeze; use a new run directory."
    assert (RUN / "pretest_freeze_audit.json").exists(), "Existing test data lacks this notebook's pretest audit record."
    print("Resuming an already frozen run; test caches already exist:", counts["test"])
else:
    print("Train/dev extraction complete; no test scene has been encoded.")
print(counts)

## 5. Train all six correction heads and the independent probe

The two correction arms use the same architecture, initialization per paired seed, and training order. `learned_residual` sees token content and a shared source appearance anchor; `geometry_only` removes content from the correction network while retaining the same deterministic residual base. It is not a completely content-blind editor.

Each arm runs 30 epochs for seeds 1801/1802/1803. Development error selects its checkpoint. The token-only occupancy/RGB readout is fitted on genuine training embeddings and selected on genuine development embeddings, then frozen. No edited features train this readout. All seven selected checkpoints are retained. Completed individual heads can be resumed only with exact source, policy, feature-binding, and checkpoint hashes.

In [ ]:
run_logged([sys.executable, "-u", EXP / "train.py", "--features", FEATURES,
            "--out", TRAINING, "--device", DEVICE], "training")
assert FREEZE.is_file(), "Training did not produce its pretest checkpoint freeze."

## 6. Display and record the pretest freeze

This is the boundary between development and held-out evaluation. The following cell verifies and displays the immutable checkpoint hashes and records their freeze hash before publication verification and any test encoding. A local audit alone is insufficient: the next section must verify the same freeze in a published Git commit. Do not modify the model, training policy, or evaluation rules after this boundary and then describe that same test set as untouched. A changed experiment needs a new protocol and new held-out scenes.

In [ ]:
frozen = json.loads(FREEZE.read_text())
assert frozen["test_accessed"] is False
assert len(frozen["models"]) == 7
for name, digest in frozen["source_hashes"].items():
    assert sha256(EXP / name) == digest, f"Changed frozen source: {name}"
for item in frozen["models"]:
    assert sha256(TRAINING / item["path"]) == item["sha256"], f"Changed checkpoint: {item['path']}"
freeze_hash = sha256(FREEZE)
print("PRETEST_FREEZE_SHA256", freeze_hash)
for item in frozen["models"]:
    print(item["arm"], "seed", item["seed"], "epoch", item["epoch"], "sha256", item["sha256"])
audit_path = RUN / "pretest_freeze_audit.json"
audit = {"recorded_utc": datetime.now(timezone.utc).isoformat(), "source_revision": CODE_COMMIT,
         "checkpoint_freeze_sha256": freeze_hash, "test_encoded_before_record": False,
         "freeze_file": str(FREEZE.relative_to(RUN)), "models": frozen["models"]}
if audit_path.exists():
    existing = json.loads(audit_path.read_text())
    assert existing["checkpoint_freeze_sha256"] == freeze_hash
    assert existing["source_revision"] == CODE_COMMIT and existing["test_encoded_before_record"] is False
    print("Existing pretest audit preserved:", existing["recorded_utc"])
else:
    current_manifest = json.loads((FEATURES / "manifest.json").read_text())
    assert not any(row["spec"]["split"] == "test" for row in current_manifest["clips"])
    audit_path.write_text(json.dumps(audit, indent=2) + "\n")
    print("Pretest audit saved. Publish this exact freeze and verify its Git commit in the next section.")

## 7. Verify the published freeze before opening test scenes

The protocol requires **Git publication of the completed checkpoint freeze before test access**. Publish the exact `training/checkpoint_freeze.json` displayed above as a repository file, then set `FREEZE_COMMIT` to that publication's full 40-character commit SHA and `FREEZE_REPO_PATH` to its path within the repository. For a fresh reproduction, publish the new run's freeze; do not substitute the historical freeze merely to proceed.

This cell fetches that commit from the configured repository, reads the published file directly with `git show`, and compares its exact bytes and SHA-256 with the local freeze. It also records the commit and path before test extraction. **The test cell refuses to run unless this verification succeeds.** This is the experiment's publication gate; no credentials or tokens are entered into this notebook, and it does not publish changes on your behalf.

In [ ]:
assert re.fullmatch(r"[0-9a-f]{40}", FREEZE_COMMIT), (
    "Publish this run's completed checkpoint_freeze.json first, then set FREEZE_COMMIT to its full Git commit SHA."
)
freeze_repo_parts = Path(FREEZE_REPO_PATH).parts
assert FREEZE_REPO_PATH and not Path(FREEZE_REPO_PATH).is_absolute() and ".." not in freeze_repo_parts
actual_origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
assert actual_origin == REPO_URL, "Publication verification must use the configured project repository."
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", FREEZE_COMMIT], check=True, env=CHILD_ENV)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", FREEZE_COMMIT + "^{commit}"], text=True).strip()
assert actual_commit == FREEZE_COMMIT
published_bytes = subprocess.check_output(["git", "-C", str(REPO), "show", f"{FREEZE_COMMIT}:{FREEZE_REPO_PATH}"])
local_bytes = FREEZE.read_bytes()
published_hash = hashlib.sha256(published_bytes).hexdigest()
assert published_bytes == local_bytes, (
    "The published freeze does not match this run. Publish this run's exact freeze and set its own commit/path."
)
assert published_hash == sha256(FREEZE)
local_audit = json.loads((RUN / "pretest_freeze_audit.json").read_text())
assert local_audit["checkpoint_freeze_sha256"] == published_hash
verification_path = RUN / "published_freeze_verification.json"
verification = {
    "verified_utc": datetime.now(timezone.utc).isoformat(), "repository": REPO_URL,
    "freeze_commit": FREEZE_COMMIT, "freeze_repo_path": FREEZE_REPO_PATH,
    "checkpoint_freeze_sha256": published_hash, "test_encoded_before_verification": False,
    "verification": "Exact byte equality to git show of the commit fetched from repository origin",
}
if verification_path.exists():
    existing = json.loads(verification_path.read_text())
    for key in ("repository", "freeze_commit", "freeze_repo_path", "checkpoint_freeze_sha256"):
        assert existing[key] == verification[key], f"Existing publication record differs: {key}"
    assert existing["test_encoded_before_verification"] is False
    print("Existing pretest Git verification preserved:", existing["verified_utc"])
else:
    current_manifest = json.loads((FEATURES / "manifest.json").read_text())
    assert not any(row["spec"]["split"] == "test" for row in current_manifest["clips"]), (
        "Test caches already exist without this run's Git publication verification; use a fresh run."
    )
    verification_path.write_text(json.dumps(verification, indent=2) + "\n")
print("VERIFIED_PRETEST_GIT_COMMIT", FREEZE_COMMIT)
print("VERIFIED_PRETEST_GIT_PATH", FREEZE_REPO_PATH)
print("VERIFIED_PRETEST_FREEZE_SHA256", published_hash)

## 8. Held-out evaluation begins here

Encode the 16 test scenes only after the local freeze and Git publication verification above. The extractor verifies the frozen source and checkpoint hashes and binds the test manifest to that freeze. The eight familiar-magnitude and eight held-out-magnitude scenes are all retained. Test data cannot be used to select a checkpoint, seed, scene, or display frame.

In [ ]:
publication_record = json.loads((RUN / "published_freeze_verification.json").read_text())
assert publication_record["test_encoded_before_verification"] is False
assert publication_record["repository"] == REPO_URL
assert publication_record["freeze_commit"] == FREEZE_COMMIT
assert publication_record["freeze_repo_path"] == FREEZE_REPO_PATH
assert publication_record["checkpoint_freeze_sha256"] == sha256(FREEZE)
# Recheck the fetched immutable Git object as well as the local audit record.
assert subprocess.check_output(["git", "-C", str(REPO), "show", f"{FREEZE_COMMIT}:{FREEZE_REPO_PATH}"]) == FREEZE.read_bytes()
assert (RUN / "pretest_freeze_audit.json").is_file()
assert json.loads((RUN / "pretest_freeze_audit.json").read_text())["checkpoint_freeze_sha256"] == sha256(FREEZE)
run_logged([sys.executable, "-u", EXP / "extract.py", "--out", FEATURES,
            "--upstream", UPSTREAM, "--splits", "test", "--freeze", FREEZE,
            "--device", DEVICE, "--precision", PRECISION], "held_out_extraction")
feature_manifest = json.loads((FEATURES / "manifest.json").read_text())
assert sum(row["spec"]["split"] == "test" for row in feature_manifest["clips"]) == 16
assert feature_manifest["test_freeze_sha256"] == sha256(FREEZE)

## 9. Score interventions and inspect diagnostic displays

Latent error is reported separately at the vacated source, destination, halo, distractor, background, and a common region covering both shift directions. No edit has normalized error 1. Three learned seeds are averaged within each scene before bootstrapping the 16 independent scenes.

The frozen probe must first work on genuine test targets. Its position and coarse colour-identity gates are reported separately; a failed probe gate limits semantic interpretation. The plots and videos show **coarse probe outputs, not generated full-resolution video**. The display scenes (11200 and 11208), model seed (1801), and still frame (15, zero-based) were fixed in the visualizer before test access.

In [ ]:
run_logged([sys.executable, "-u", EXP / "evaluate.py", "--features", FEATURES,
            "--run", TRAINING, "--out", TEST, "--device", DEVICE], "test_evaluation")
run_logged([sys.executable, "-u", EXP / "visualize.py", "--test", TEST, "--out", ANALYSIS], "scientific_visualization")
summary = json.loads((TEST / "summary.json").read_text())
print(json.dumps({key: summary[key] for key in ("n_independent_scenes", "genuine_target_readout_gate", "genuine_target_identity_gate", "decision_checks") if key in summary}, indent=2))
from IPython.display import display, Image as DisplayImage, Video
for path in sorted(ANALYSIS.glob("*.png")):
    display(DisplayImage(filename=str(path)))
for path in sorted(ANALYSIS.glob("*_comparison.mp4")):
    display(Video(filename=str(path), embed=True))

## 10. Preserve code, checkpoints, metrics, predictions, and logs

The archive includes all trained heads and the frozen probe, the pretest freeze/audit and Git publication verification, exact CSV/NPZ test evidence, displays, package/runtime records, feature metadata, and the pinned experiment source. The official backbone checkpoint is independently downloadable and hash-verified, so it is not duplicated. Reproducible feature caches are omitted by default; set `INCLUDE_FEATURE_CACHE = True` to include them too.

Download the ZIP before a Colab runtime is discarded. The notebook itself remains a clean source recipe; save an executed notebook separately if execution outputs are desired. Publishing the resulting archive and freeze record alongside the experiment's repository revision preserves the actual evidence.

In [ ]:
INCLUDE_FEATURE_CACHE = False
ARCHIVE = BASE_DIR / f"{RUN_NAME}_evidence.zip"
source_snapshot = RUN / "source" / "experiments" / "day8"
source_snapshot.mkdir(parents=True, exist_ok=True)
for path in EXP.iterdir():
    if path.is_file() and path.suffix in (".py", ".md", ".ipynb", ".json"):
        shutil.copy2(path, source_snapshot / path.name)
archive_policy = {"include_feature_cache": INCLUDE_FEATURE_CACHE,
                  "backbone_checkpoint_included": False,
                  "source_revision": CODE_COMMIT,
                  "checkpoint_freeze_sha256": sha256(FREEZE),
                  "freeze_publication_commit": FREEZE_COMMIT,
                  "freeze_publication_repo_path": FREEZE_REPO_PATH,
                  "feature_manifest_sha256": sha256(FEATURES / "manifest.json"),
                  "test_summary_sha256": sha256(TEST / "summary.json"),
                  "created_utc": datetime.now(timezone.utc).isoformat()}
(RUN / "archive_policy.json").write_text(json.dumps(archive_policy, indent=2) + "\n")
with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RUN.rglob("*")):
        if not path.is_file() or "__pycache__" in path.parts:
            continue
        if not INCLUDE_FEATURE_CACHE and FEATURES / "cache" in path.parents:
            continue
        archive.write(path, path.relative_to(RUN))
with zipfile.ZipFile(ARCHIVE) as archive:
    assert archive.testzip() is None
archive_record = {"file": str(ARCHIVE), "bytes": ARCHIVE.stat().st_size, "sha256": sha256(ARCHIVE), **archive_policy}
ARCHIVE.with_suffix(".zip.sha256.json").write_text(json.dumps(archive_record, indent=2) + "\n")
print(json.dumps(archive_record, indent=2))
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(str(ARCHIVE)))
else:
    files.download(str(ARCHIVE))